# Analog Perceptron -- the math

This is the math for a single 2-input, trainable perceptron, meant to be built as real analog hardware (the rest of that project -- ESP32 training loop, power supply, level-shifting between the 3.3V digital side and the analog rails -- lives elsewhere).

**Scope, on purpose:** the analog hardware computes exactly one thing -- the weighted sum `y = w1*x1 + w2*x2 + b`. The activation function (a threshold) and the learning rule both live in software on the microcontroller side, not in analog circuitry. That keeps the physical build to the same two-op-amp summing-amplifier topology from the `01-getting-started` notebook, just with two weighted inputs instead of one, and with `w1`, `w2`, `b` meant to be driven by digital potentiometers (trainable) rather than fixed resistors.

(A hand-rolled schematic exporter lived here briefly -- dropped it in favor of pointing this `State` at an actual open-source circuit tool instead of reinventing schematic layout.)

In [1]:
from dda import State, symbols, dda, export

x1, x2, w1, w2, b, y = symbols("x1, x2, w1, w2, b, y")

## The math

Same shape as `y = m*x + b`, just with a second weighted input added into the summing junction before the inverter.

In [2]:
s = State()
s[y] = dda.neg(dda.sum(dda.mult(w1, x1), dda.mult(w2, x2), b))

s

State({'y': neg(sum(mult(w1, x1), mult(w2, x2), b))})

In [3]:
export(s, to="sympy")  # sanity check: y = w1*x1 + w2*x2 + b, no sign errors

[Eq(y, b + w1*x1 + w2*x2)]

## A real circuit, not a picture

`export(state, to="falstad")` (dropped in a fork of anabrid's `pyanalog`) turns this `State` into an actual [Falstad](https://www.falstad.com/circuit/) circuit -- real op-amps, real resistor values, wired and ready to simulate, not a diagram of one.

In [4]:
circuit = export(s, to="falstad")
print("unsupported:", circuit.unsupported)
circuit.url()

unsupported: ['mult_1', 'mult_2']


'https://www.falstad.com/circuit/circuitjs.html?cct=%24%201%205.0E-6%2010%2057%2015.0%2050%0AR%2096%2096%2048%2096%200%200%2040.0%202.0%200.0%0Ax%20-42%2072%20-18%2088%200%2014%20b%0AR%2096%20208%2048%20208%200%200%2040.0%202.0%200.0%0Ax%20-42%20184%20-10%20200%200%2014%20w1%0AR%2096%20320%2048%20320%200%200%2040.0%202.0%200.0%0Ax%20-42%20296%20-10%20312%200%2014%20w2%0AR%2096%20432%2048%20432%200%200%2040.0%202.0%200.0%0Ax%20-42%20408%20-10%20424%200%2014%20x1%0AR%2096%20544%2048%20544%200%200%2040.0%202.0%200.0%0Ax%20-42%20520%20-10%20536%200%2014%20x2%0Ax%201088%2096%201936%20112%200%2014%20mult_1%3A%5C%20no%5C%20analog%5C%20multiplier%5C%20in%5C%20CircuitJS1%5C%20for%5C%20mult_1%5C%20%3D%5C%20w1%2Ax1%5C%20--%5C%20wire%5C%20a%5C%20VCCS%5C%20with%5C%20expression%5C%20a%2Ab%5C%20by%5C%20hand%0Ax%201088%20272%201936%20288%200%2014%20mult_2%3A%5C%20no%5C%20analog%5C%20multiplier%5C%20in%5C%20CircuitJS1%5C%20for%5C%20mult_2%5C%20%3D%5C%20w2%2Ax2%5C%20--%5C%20wire%5C%20a%5C%20VCCS%5C%20wi

`unsupported` isn't empty here, and that's the honest result, not a bug: `w1` and `w2` are live variables (they're meant to be set by a digital pot during training), so `mult(w1, x1)` and `mult(w2, x2)` both need an actual four-quadrant analog multiplier IC -- and CircuitJS1 doesn't have one as a built-in part. The link still opens a real, correctly-wired circuit for everything *except* those two multiply stages (the summing junction and output inverter are there and correct); the multiplier inputs show up as a labeled gap you'd wire an AD633, or a VCCS with a custom `a*b` expression, into by hand.

For a circuit that's *fully* realizable start to finish, see the decay/oscillator examples in `01-getting-started.ipynb` -- those have no live-signal multiplies, so `unsupported` comes back empty and the whole thing loads and runs as-is.